# RL Challenge – PPO-Baseline trainieren

Dieses Notebook klont immer die aktuelle Repository-Version, trainiert die einfache PPO-Baseline als Spieler 1 gegen einen zufälligen Gegner und exportiert eine turnierkompatible Datei.

Vor dem Start: **Runtime → Change runtime type → T4 GPU** auswählen. Ohne GPU funktioniert das Notebook ebenfalls, aber deutlich langsamer.

In [ ]:
# Alte Kopie löschen: Colab-Speicher ist nur temporär und wird dadurch aktuell.
%cd /content
!rm -rf /content/rl-challenge
# Repository und alle benötigten Python-Pakete installieren.
!git clone --depth 1 https://github.com/georgmerz/rl-challenge.git
%cd rl-challenge
!pip install -q -r requirements.txt
# PettingZoo liefert die Atari-ROMs nicht mit. AutoROM installiert sie einmalig.
!AutoROM --accept-license

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch-Version: {torch.__version__}')
print(f'Verwendetes Gerät: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('Hinweis: Für schnelleres Training in Colab eine T4-GPU aktivieren.')

In [ ]:
# Parameter: Für einen ersten Test genügen 20_000 Schritte.
# 100_000 Schritte sind der normale längere Trainingslauf.
TIMESTEPS = 100_000
OUTPUT = 'baselines/baseline_ppo.pt2'

!python train_ppo_sb3.py --device {device} --timesteps {TIMESTEPS} --output {OUTPUT}

In [ ]:
# Prüfen, dass der portable PPO-Export tatsächlich geladen werden kann.
from pathlib import Path
import torch

assert Path(OUTPUT).is_file(), f'Export fehlt: {OUTPUT}'
policy = torch.jit.load(OUTPUT, map_location='cpu').eval()
example_action = policy(torch.zeros((1, 3, 210, 160)), torch.ones((1,), dtype=torch.int64))
print(f'Export geprüft: {OUTPUT} · Beispielaktion: {int(example_action.item())}')

In [ ]:
# Die fertige, turnierkompatible Policy herunterladen.
from google.colab import files
files.download(OUTPUT)